# Code to generate 3D mesh of an oblate spheroidal hailstone with roughly equidistant lobes

Authors: Becky Adams-Selin and Chase Calkins, AER

In [1]:
import numpy as np
from matplotlib import cm
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import pyvista as pv
import trimesh
from copy import deepcopy
import pymeshfix as mf
from geographiclib.geodesic import Geodesic
%matplotlib widget

### Set oblate spheroidal and lobe properties as desired

In [2]:
#Number of lobes you want on your hailstone.
n_lobes = 100 

#How narrow you want your lobes to be. Decimal number between
# 0 (lobe nonexistent, no point!). to 0.45 (as large as possible 
# without overlap) to something higher (at 1.0, they will fully
# overlap with the neighboring node). Note lobe width is also 
# dependent on the number of lobes, and the sine/cosine equation 
# used to generate them. That equation would need to be modified 
# in code below.
lobe_factor = 0.75

#Lobe height factor. Setting it to 1 will make the lobes a factor
# of 1.5*radius. Setting it to 0 will make the lobes the same
# level as the radius (non-existent).
height_factor = 0.25

#Radii of the oblate spheroid underlying the lobes.
radius_a = 50. #mm, major/maximum dimension, assumed to be horizontal x dimension
radius_c = radius_a #other horizontal axis (y). They are assumed to be equal here.
radius_b = 40. #mm minor/minimum dimension, assumed to be vertical z dimension.

#create a geodesic instance with these dimensions
f = 1 - radius_b / radius_a #"flattening" of the spheroid
geod = Geodesic(radius_a, f)

### Define a function to fix the normals of both the sphere and lobes (courtesy of Joshua Soderholm)

In [3]:
def recalculate_normals_from_centroid(mesh):
    """
    Recalculate all normals to point outward from the mesh centroid.
    Handles triangular, quad, and mixed polygon meshes.
    
    Parameters
    ----------
    mesh : pv.PolyData
        The mesh to process
        
    Returns
    -------
    mesh : pv.PolyData
        The mesh with corrected normals
    """
    # Get the centroid of the entire mesh
    centroid = mesh.center
    
    # Get cell centers
    cell_centers = mesh.cell_centers().points
    
    # Calculate vectors from centroid to each cell center
    outward_vectors = cell_centers - centroid
    
    # Normalize these vectors to get desired normal directions
    desired_normals = outward_vectors / np.linalg.norm(outward_vectors, axis=1, keepdims=True)
    
    # Compute the current normals
    mesh.compute_normals(cell_normals=True, point_normals=False, inplace=True)
    current_normals = mesh.cell_data['Normals']
    
    # Check which normals need flipping
    dot_products = np.sum(current_normals * desired_normals, axis=1)
    needs_flipping = dot_products < 0
    
    # Process faces - need to handle variable polygon sizes
    faces = mesh.faces.copy()
    n_cells = mesh.n_cells
    
    # Parse faces array and flip as needed
    idx = 0
    for cell_id in range(n_cells):
        # First value is number of points in this face
        n_points = faces[idx]
        
        # Get the vertex indices for this face
        vertex_indices = faces[idx+1:idx+1+n_points]
        
        # If this face needs flipping, reverse the vertex order
        if needs_flipping[cell_id]:
            faces[idx+1:idx+1+n_points] = vertex_indices[::-1]
        
        # Move to next face
        idx += n_points + 1
    
    # Update the mesh faces
    mesh.faces = faces
    
    # Recompute normals after flipping
    mesh.compute_normals(cell_normals=True, point_normals=True, consistent_normals=True)
    
    print(f"Flipped {needs_flipping.sum()} out of {len(needs_flipping)} faces")
    
    return mesh

### Function to fix the small holes that make the shape not watertight

In [4]:
def fill_all_hole_loops(mesh, holes):
    """
    Fill all hole loops on a triangulated PyVista mesh by creating fan triangles
    from each loop's centroid to its boundary vertices.
    
    Parameters
    ----------
    mesh : pyvista.PolyData
        Triangulated input mesh (will not be modified in-place).
    holes : pyvista.PolyData
        Lines polydata returned by MeshFix.extract_holes() (edge format).
    
    Returns
    -------
    pyvista.PolyData
        New mesh with all holes filled (triangles appended).
    """
    import numpy as np
    import pyvista as pv
    
    if holes is None or holes.n_points == 0 or holes.n_cells == 0:
        return mesh.copy()
    
    # Parse holes.lines to extract edges (2-point line segments)
    lines = holes.lines.astype(int)
    edges = []
    idx = 0
    while idx < len(lines):
        n = int(lines[idx])
        if n == 2:  # edge
            p0, p1 = lines[idx+1], lines[idx+2]
            edges.append((p0, p1))
        idx += n + 1
    
    if not edges:
        return mesh.copy()
    
    # Reconstruct closed loops from edges
    # Build adjacency: for each point, what points does it connect to?
    from collections import defaultdict
    adj = defaultdict(list)
    for p0, p1 in edges:
        adj[p0].append(p1)
        adj[p1].append(p0)
    
    # Extract loops by following edges
    loops = []
    visited_edges = set()
    
    for start_p, neighbors in adj.items():
        for next_p in neighbors:
            edge_key = (min(start_p, next_p), max(start_p, next_p))
            if edge_key in visited_edges:
                continue
            
            # Trace loop starting from this edge
            loop = [start_p, next_p]
            current = next_p
            prev = start_p
            visited_edges.add(edge_key)
            
            while current != start_p:
                # Find next neighbor (not the one we came from)
                next_neighbors = [p for p in adj[current] if p != prev]
                if not next_neighbors:
                    break  # Dead end, not a closed loop
                
                next_p = next_neighbors[0]
                edge_key = (min(current, next_p), max(current, next_p))
                visited_edges.add(edge_key)
                
                if next_p == start_p:
                    # Loop closed
                    if len(loop) >= 3:
                        loops.append(np.array(loop))
                    break
                else:
                    loop.append(next_p)
                    prev = current
                    current = next_p
    
    if not loops:
        print("No closed loops found in hole edges")
        return mesh.copy()
    
    print(f"Found {len(loops)} hole loops")
    
    # Copy original mesh data
    orig_pts = mesh.points.copy()
    orig_faces = mesh.faces.astype(int)
    
    # Convert existing faces to explicit triangle list. Not necessary if already triangulated.
    tris = []
    j = 0
    while j < len(orig_faces):
        nv = int(orig_faces[j])
        verts = orig_faces[j + 1: j + 1 + nv].tolist()
        if nv == 3:
            tris.append(verts)
        else:
            # fan-triangulate polygon
            for k in range(1, nv - 1):
                tris.append([verts[0], verts[k], verts[k + 1]])
        j += nv + 1
    
    tris = np.array(tris, dtype=int)
    
    # Start with original mesh points, then append hole boundary points and centroids
    new_points = orig_pts.tolist()
    new_tris = tris.tolist()
    
    # Map from holes.points indices to new_points indices
    hole_pts_offset = len(new_points)
    for pt in holes.points:
        new_points.append(pt.tolist())
    
    # Fill each hole loop with fan triangles to its centroid
    for loop_idx, loop in enumerate(loops):
        # loop contains indices into holes.points
        # Map them to indices in new_points
        loop_indices_in_new = [hole_pts_offset + idx for idx in loop]
        
        # Compute centroid from actual hole boundary points
        coords = holes.points[loop]
        centroid = coords.mean(axis=0)
        centroid_idx = len(new_points)
        new_points.append(centroid.tolist())
        
        # Create fan triangles: (loop[i], loop[i+1], centroid) for each edge
        L = len(loop_indices_in_new)
        
        for k in range(L):
            i0 = loop_indices_in_new[k]
            i1 = loop_indices_in_new[(k + 1) % L]
            new_tris.append([i0, i1, centroid_idx])
    
    # Build VTK faces array from triangles: [3, i0, i1, i2, 3, j0, j1, j2, ...]
    new_tris = np.array(new_tris, dtype=np.int64)
    
    # Create faces array: prepend count (3) to each triangle
    faces_list = []
    for tri in new_tris:
        faces_list.append(3)
        faces_list.extend(tri)
    
    faces_vtk = np.array(faces_list, dtype=np.int64)
    
    # Create and return new mesh
    new_pts_arr = np.array(new_points, dtype=float)
    print(f"Created mesh with {len(new_tris)} triangles, {len(new_pts_arr)} points")
    new_mesh = pv.PolyData(new_pts_arr, faces_vtk)
    
    # Recompute normals 
    new_mesh.compute_normals(cell_normals=True, point_normals=True, consistent_normals=True)
    
    return new_mesh

### Add equally spaced lobes to the oblate spheroidal surface

In [40]:
# Create a range of equidistant points on an oblate spheroidal surface. 
# This follows the same framework as for a sphere via Deserno 2004, but
# is modified to account for non-sphericity.

# A few differences from the spherical notebook. lambda is the longitudinal angle (0 to 2pi). 
# phi is the geodetic latitude - not the same as geocentric latitude. 
# It runs from -pi/2 to pi/2. 
# Recommended reading:
#  https://www.fatiando.org/harmonica/v0.7.0/user_guide/coordinate_systems.html
#  https://jerrymahun.com/index.php/home/open-access/93-updating/428-chapter-c-ellipsoid


#create a random number generator
rng = np.random.default_rng() #will generate b/w 0 and 1

lobe_centers = [] #just the center points in cartesian space
lobes = [] #successive sets of points, including circles around center points, one for each lobe
lobe_count = 0 #actual number of lobes
lobe_lambdas = []
lobe_phis = []
lobe_rpvs = [] #radius of "prime vertical", line from z-axis to the point normal to surface of 
               #spheroid at point phi, lambda. Angle between z-axis and the rpv is phi.

#get the coordinates of all lobe centers

#determine non-lobey shape surface area
e = (1-(radius_b**2./radius_a**2))**0.5 #eccentricity
sa = 2*np.pi*radius_a**2 + np.pi*(radius_b**2/e)*np.log((1+e)/(1-e))

#find average distance (d) between points.
a = sa / n_lobes
d = np.sqrt(a)

#calculate number of points in N/S direction, along a meridian (pole-pole line at constant lambda).
#ellipsoid circumference along half of the meridian, using an approximation:
c_meridian = (np.pi/2) * (3*(radius_a+radius_b) - ((3*radius_a+radius_b)*(radius_a+3*radius_b))**0.5)
#given average distance b/w points, get number of points 
M_phi = np.round(c_meridian/d) 
#and now distance between the points along the merdian
d_phi = c_meridian / M_phi

#given that distance, what is our distance between points along the constant latitude circle?
d_lambda = a / d_phi

#for each point along the half-meridian, generate a circle of constant geodetic latitude
for m in np.arange(0,M_phi):

    #First, we generate the equally space latitudes in projected spherical space (aka authalic latitudes or beta)
    #generate a random number for starting phi, so we don't always start at 0 or d_phi
    rand = 0.5*(rng.random()-0.5) #will be between -0.25 and 0.25
    #generate beta at equally spaced areas
    beta = 0.95*np.pi*((m+0.5+rand) / M_phi - 0.5)
    #convert authalic latitudes to geodetic, via approximation
    phi = beta + (e**2/3)*np.sin(2*beta) + (3*e**4/20)*np.sin(4*beta) + (5*e**6/56)*np.sin(6*beta)
    
    #Calculate circumference of horizontal cross-section at this latitude, phi.
    #Radius in the prime vertical (line normal to the spheroidal surface at this point)
    r_pv = radius_a / ((1 - e**2*(np.sin(phi))**2)**0.5)
    #Circumference of the horizontal circle cross-section with that radius, scaled for latitude
    c_parallel = 2*np.pi * r_pv * np.cos(phi)
    
    #number of equidistant points along this circle
    M_lambda = np.round(c_parallel/d_lambda)
    
    
    #for each point along this latitude circle, generate M_lambda equidistant points
    for n in np.arange(0,M_lambda):
        #a random number for starting lambda
        rand = 0.5*(rng.random()-0.5) #will be between -0.25 and 0.25
        #equally spaced lambdas(+/- rand factor)
        lambda_angle = 2*np.pi*(n+rand)/M_lambda
        
        #add a small random factor to phi, so lobes will be slightly
        #displaced off the latitude circle of constant phi
        #first, convert d_phi, distance between latitude circles, to latitude instead of arclength
        d_phi_angle = np.pi * (d_phi / c_meridian)
        #now add random factor
        rand = 0.5*(rng.random()-0.5) #will be between -0.25 and 0.25
        new_phi = phi + d_phi_angle*rand
        
        #calculate new radius in prime vertical with this new phi
        new_r_pv = radius_a / ((1 - e**2*(np.sin(new_phi))**2)**0.5)
        #given this prime vertical radius and new phi, calculate the distance to the spheroid surface
        # (geocentric radius):
        r_surface = new_r_pv / np.sqrt(1 + (e**2 * np.cos(new_phi)**2) / (1 - e**2 * np.sin(new_phi)**2))
        
        #convert to cartesian coordinates and store
        lobe_lambdas.append(lambda_angle)
        lobe_phis.append(new_phi)
        lobe_rpvs.append(new_r_pv)
        #must use geocentral latitude when converting to cartesian
        new_phi_gc = np.arctan((1 - e**2) * np.tan(new_phi))
        x = r_surface * np.cos(new_phi_gc)*np.cos(lambda_angle)
        y = r_surface * np.cos(new_phi_gc)*np.sin(lambda_angle)
        z = r_surface * np.sin(new_phi_gc)
        lobe_centers.append((x,y,z))
        lobe_count += 1

#Check to make sure we didn't leave a gap at either pole
#convert d_phi, distance between latitude circles, to latitude instead of arclength
d_phi_angle = np.pi * (d_phi / c_meridian)

#north pole first (phi = -pi/2)
if min(lobe_phis) - (-np.pi/2) >= d_phi_angle:
    #add a new lobe, a random offset from the north pole
    rand = 0.25*rng.random() #will be between 0 and 0.125
    phi = d_phi_angle*rand - np.pi/2 #small amount offset from -pi/2
    rand = rng.random() #between 0 and 1
    lambda_angle = 2*np.pi*rand #random longitude b/w 0 and 2pi
    #given this new latitude, determine new r_pv
    r_pv = radius_a / ((1 - e**2*(np.sin(phi))**2)**0.5)
    #and corresponding r_surface
    r_surface = r_pv / np.sqrt(1 + (e**2 * np.cos(phi)**2) / (1 - e**2 * np.sin(phi)**2))

    #calculate cartesian parameters and store    
    lobe_lambdas.append(lambda_angle)
    lobe_phis.append(phi)
    lobe_rpvs.append(r_pv)
    #must use geocentral latitude when converting to cartesian
    phi_gc = np.arctan((1 - e**2) * np.tan(new_phi))
    x = r_surface * np.cos(phi_gc)*np.cos(lambda_angle)
    y = r_surface * np.cos(phi_gc)*np.sin(lambda_angle)
    z = r_surface * np.sin(phi_gc)
    lobe_centers.append((x,y,z))
    lobe_count += 1    
    
#and check again for the south pole (phi = pi/2)
if np.pi/2 - max(lobe_phis) >= d_phi_angle:
    #add a new lobe, a random offset from the south pole
    rand = 0.25*rng.random() #will be between 0 and 0.125
    phi = np.pi/2 - d_phi_angle*rand  #small amount offset from pi/2
    rand = rng.random() #between 0 and 1
    lambda_angle= 2*np.pi*rand #random longitude b/w 0 and 2pi
    #given this new latitude, determine new r_pv
    r_pv = radius_a / ((1 - e**2*(np.sin(phi))**2)**0.5)
    #and corresponding r_surface
    r_surface = r_pv / np.sqrt(1 + (e**2 * np.cos(phi)**2) / (1 - e**2 * np.sin(phi)**2))
    
    #calculate cartesian parameters and store    
    lobe_lambdas.append(lambda_angle)
    lobe_phis.append(phi)
    lobe_rpvs.append(r_pv)
    #must use geocentral latitude when converting to cartesian
    phi_gc = np.arctan((1 - e**2) * np.tan(new_phi))    
    x = r_surface * np.cos(phi_gc)*np.cos(lambda_angle)
    y = r_surface * np.cos(phi_gc)*np.sin(lambda_angle)
    z = r_surface * np.sin(phi_gc)
    lobe_centers.append((x,y,z))
    lobe_count += 1    


# Add this diagnostic after generating lobe_centers but before the lobe circles

lobe_centers_np = np.array(lobe_centers)
print("=== LOBE CENTERS AXIS RANGES ===")
print(f"x range: [{lobe_centers_np[:,0].min():.2f}, {lobe_centers_np[:,0].max():.2f}]")
print(f"y range: [{lobe_centers_np[:,1].min():.2f}, {lobe_centers_np[:,1].max():.2f}]")
print(f"z range: [{lobe_centers_np[:,2].min():.2f}, {lobe_centers_np[:,2].max():.2f}]")

print("\n=== EXPECTED (from PyVista) ===")
print(f"x radius: {radius_a}")
print(f"y radius: {radius_c}")
print(f"z radius: {radius_b}")

print("\n=== GEOD PARAMETERS ===")
print(f"Geodesic semi-major axis: {radius_a}")
print(f"Geodesic flattening: {f}")
print(f"Implied semi-minor axis: {radius_a * (1 - f)}")

#convert d_lambda to angle between longitude points instead of distance
c_azimuth = 2*np.pi*radius_a #azimuthal (e-w) cross-section is just a circle since radius_a=radius_c
d_lambda_angle = np.pi * (d_lambda / c_azimuth)

#now calculate "circles" around each lobe point, starting at a central angle of alpha 
# from each lobe point
alpha_start = lobe_factor*np.minimum(d_phi_angle, d_lambda_angle)
num_t_at_start = 40 #originally 40, but that looked a little choppy when printed
circ_at_start = 2*np.pi*alpha_start
dt = circ_at_start / num_t_at_start
#lobe_heights = radius + np.sin(np.pi/2 * np.linspace(0,0.5,10)) #equation too sharp
#The below uses https://easings.net/#easeInOutSine instead
# lobe_heights = 1+height_factor*\
#                 (-(np.cos(np.pi*np.linspace(0,1.0,num_t_at_start))-1)/4)   
lobe_heights = np.ones(num_t_at_start)

for phi, lambda_angle, lobe_center, r_pv_center in zip(lobe_phis, lobe_lambdas, lobe_centers, lobe_rpvs):
    lobe_points = []
    
    #cycle through successively smaller alphas, setting the radius/height increasingly higher
    # each time
    alphas = np.linspace(alpha_start, 0, num_t_at_start)
    #get distance to the surface from r_pv
    r_surface = r_pv_center / np.sqrt(1 + (e**2 * np.cos(phi)**2) / (1 - e**2 * np.sin(phi)**2))

                    
    xs, ys, zs = [], [], []   #3d cartesian points for lobes               
                
    for alpha, lobe_height in zip(alphas[:-1], lobe_heights[:-1]):
    
        #calculate number of t needed for this alpha
        circ_here = 2*np.pi*alpha
        num_t = int(np.round(circ_here/dt))
        t = np.linspace(0,2*np.pi,num_t)
        
        #find the distance between center point (phi, lambda) and point alpha angle away.
        #to do that, calculate the radius of curvature in both the prime vertical and meridional
        # directions. 
        #we already know r_pv at this point, it is r_pv_center
        # meridional radius is from https://jerrymahun.com/index.php/home/open-access/93-updating/428-chapter-c-ellipsoid?start=3
        r_m = (radius_a*(1-e**2.))/ (1 - e**2.*np.sin(phi)**2)**0.5
        r_mean = np.sqrt(r_m * r_pv_center)
        #distance = r_mean * alpha # should be in mm
        distance = r_pv * alpha


        #determine lat,lon of all the points in the circle
        for azimuth in t: #all the angles/headings about the circle
        
            #the function below can't handle negative longitudes. 
            if lambda_angle < 0:
                lambda_angle = 2*np.pi + lambda_angle
            
            # Use geographiclib to find point at this distance and azimuth
            #print (np.degrees(phi), np.degrees(lambda_angle), np.degrees(azimuth), distance)
            result = geod.Direct(np.degrees(phi), np.degrees(lambda_angle), np.degrees(azimuth), distance)
            lat, lon = result['lat2'], result['lon2']
            #print (lat, lon)
            
            # Convert back to radians
            lat_rad = np.radians(lat)
            lon_rad = np.radians(lon)
            
            # Calculate r_pv at this new circle point location
            r_pv_point = radius_a / ((1 - e**2*(np.sin(lat_rad))**2)**0.5)
            # and from there, calculate the geocentric radius (distance from center to surface)
            r_surface = r_pv_point / \
                        np.sqrt(1 + (e**2 * np.cos(lat_rad)**2) / (1 - e**2 * np.sin(lat_rad)**2))
            
            #must use geocentral latitude when converting to cartesian
            lat_gc = np.arctan((1 - e**2) * np.tan(lat_rad))
            
            #and now get the cartesian coordinates for these points
            x = lobe_height * r_surface * np.cos(lat_gc) * np.cos(lon_rad)
            y = lobe_height * r_surface * np.cos(lat_gc) * np.sin(lon_rad)
            z = lobe_height * r_surface * np.sin(lat_gc)

            #now store these
            xs.append(x)
            ys.append(y)
            zs.append(z)
            lobe_points.append((x,y,z))


    #last point is right at alpha = 0
    alpha = alphas[-1]
    lobe_height = lobe_heights[-1]
    #calculate r_surface at lobe center
    r_surface = r_pv / np.sqrt(1 + (e**2 * np.cos(phi)**2) / (1 - e**2 * np.sin(phi)**2))

    #must use geocentral latitude when converting to cartesian
    phi_gc = np.arctan((1 - e**2) * np.tan(phi))

    #use the x,y,z location of lobe center but add lobe_height
    x = lobe_height * r_surface * np.cos(phi_gc) * np.cos(lambda_angle)
    y = lobe_height * r_surface * np.cos(phi_gc) * np.sin(lambda_angle)
    z = lobe_height * r_surface * np.sin(phi_gc)
    #now store these
    xs.append(x)
    ys.append(y)
    zs.append(z)
    lobe_points.append((x,y,z))
    #print (alpha, ': ', len(xs), len(ys), len(zs), len(lobe_points))
   
    lobes.append(lobe_points)

print ('len of lobes', len(lobes))
print ('len of lobe centers', len(lobe_centers))


=== LOBE CENTERS AXIS RANGES ===
x range: [-42.88, 43.06]
y range: [-43.06, 42.76]
z range: [-60.07, 61.61]

=== EXPECTED (from PyVista) ===
x radius: 50.0
y radius: 50.0
z radius: 40.0

=== GEOD PARAMETERS ===
Geodesic semi-major axis: 50.0
Geodesic flattening: 0.19999999999999996
Implied semi-minor axis: 40.0
len of lobes 109
len of lobe centers 109


### Reconstruct a surface by merging lobe points in the point cloud to an oblate spheroid

In [39]:
#Use pyvista to generate an oblate spheroid. Increase u_res, v_res, w_res will increase file size.
#https://docs.pyvista.org/api/utilities/_autosummary/pyvista.parametricellipsoid
spheroid_surf = pv.ParametricEllipsoid(xradius = radius_a, yradius=radius_c, zradius=radius_b)

#calculate a separate surface for each lobe
lobe_surfs = []
for lobe in lobes:
    lobe_points = np.array(lobe)
    #create a point cloud from each set of lobe points
    lobe_point_cloud = pv.PolyData(lobe_points)
    #lobe_surfs.append(lobe_point_cloud.reconstruct_surface(sample_spacing=0.05))
    #triangulate our lobes instead
    lobe_surf = lobe_point_cloud.delaunay_2d()
    lobe_surfs.append(lobe_surf)
print('done with lobe for loop')

#add our surfaces together
#sphere_surf = sphere_cloud.reconstruct_surface() - very memory hungry, replaced with pv.sphere above
total_surf = deepcopy(spheroid_surf)
for lobe_surf in lobe_surfs:
    #total_surf = total_surf + lobe_surf 
    #below gives fewer duplicate points to clean
    total_surf = total_surf.merge(lobe_surf)

#convert polygons to triangles
total_surf = total_surf.triangulate()    
# Fix the surface normals to make them all point outward
cell_normals_recal = recalculate_normals_from_centroid(total_surf)

# #and plot, by individual lobe and then combined surface    
pv.global_theme.color_cycler = 'default'
pl = pv.Plotter(shape=(1, 2))
pl.subplot(0, 0)
_ = pl.add_mesh(spheroid_surf, opacity=0.5)
for lobe_surf in lobe_surfs:
    _ = pl.add_mesh(lobe_surf)
    
pl.subplot(0, 1)
_ = pl.add_mesh(total_surf,color='blue')
pl.show()



done with lobe for loop
Flipped 99597 out of 178299 faces


Widget(value='<iframe src="http://localhost:60489/index.html?ui=P_0x165f39580_20&reconnect=auto" class="pyvist…

### Write mesh out to STL file

In [7]:
total_surf.save('spheroid_n'+str(int(n_lobes))+'lobe_w'+str(int(lobe_factor*100))+
                '_h'+str(int(height_factor*100))+'_radA'+str(int(radius_a))+
                '_radB'+str(int(radius_b))+'.stl')


#now plot again
mesh = trimesh.load_mesh('spheroid_n'+str(int(n_lobes))+'lobe_w'+str(int(lobe_factor*100))+
                '_h'+str(int(height_factor*100))+'_radA'+str(int(radius_a))+
                '_radB'+str(int(radius_b))+'.stl')
# Show the mesh (opens in a window)
mesh.show()


In [8]:
#mesh quality
print(f"- Is watertight: {mesh.is_watertight}")
print(f"- Volume: {mesh.volume:.4f}")
print(f"- Surface area: {mesh.area:.4f}")




- Is watertight: False
- Volume: 6691459089.0705
- Surface area: 13776407.8347


### Now actually do the hole fixing

In [9]:
tri_surf = total_surf.triangulate()
mesh_to_fix = mf.MeshFix(tri_surf)
holes = mesh_to_fix.extract_holes()

# Show holes before filling
p = pv.Plotter()
p.add_mesh(tri_surf, color='blue')
p.add_mesh(holes, color='red', line_width=8)
p.show()

# Fill all holes
filled = fill_all_hole_loops(tri_surf, holes)
print(f"Original mesh: {tri_surf.n_points} points, {tri_surf.n_cells} cells")
print(f"Filled mesh: {filled.n_points} points, {filled.n_cells} cells")

# Check if watertight (convert to trimesh to access is_watertight)
filled_trimesh = trimesh.Trimesh(vertices=filled.points, faces=filled.faces.reshape(-1, 4)[:, 1:])
print(f"Is filled mesh watertight? {filled_trimesh.is_watertight}")

# Visualize result
filled.plot()

Widget(value='<iframe src="http://localhost:60489/index.html?ui=P_0x16a4e55b0_1&reconnect=auto" class="pyvista…

Found 57 hole loops
Created mesh with 106272 triangles, 55475 points
Original mesh: 53195 points, 104049 cells
Filled mesh: 55475 points, 106272 cells
Is filled mesh watertight? True


Widget(value='<iframe src="http://localhost:60489/index.html?ui=P_0x173569be0_2&reconnect=auto" class="pyvista…

### Do a final check. Section below isn't really necessary, you already know if it's watertight.

In [10]:

# Diagnostic: analyze what holes remain
print("\n=== DIAGNOSTIC ===")
print(f"Original mesh is_watertight: {mesh.is_watertight}")
print(f"Filled mesh (trimesh) is_watertight: {filled_trimesh.is_watertight}")
print(f"Filled mesh volume: {filled_trimesh.volume:.6f}")

# Check if there are still holes in the filled mesh
mesh_to_fix_after = mf.MeshFix(filled)
holes_after = mesh_to_fix_after.extract_holes()

print(f"\nHoles before filling: {holes.n_cells} hole segments")
print(f"Holes after filling: {holes_after.n_cells} hole segments")

# The key question: is it actually watertight for practical purposes?
if filled_trimesh.is_watertight:
    print("\n✓ SUCCESS: Mesh is watertight according to trimesh!")
    print("  (The remaining holes detected by MeshFix are likely topological artifacts)")
    print("  Your mesh is ready for use.")
    
    # Save the watertight mesh
    filled_trimesh.export('equimesh_n'+str(int(n_lobes))+'lobe_w'+str(int(lobe_factor*100))+
                '_h'+str(int(height_factor*100))+'_watertight.stl')
    print('  Saved to: equimesh_n'+str(int(n_lobes))+'lobe_w'+str(int(lobe_factor*100))+
                '_h'+str(int(height_factor*100))+'_watertight.stl')
else:
    print("\n✗ Mesh is still not watertight")
    # Visualize remaining holes if any
    if holes_after.n_cells > 0:
        print("Remaining holes detected! Visualizing...")
        p2 = pv.Plotter()
        p2.add_mesh(filled, color='blue', opacity=0.7)
        p2.add_mesh(holes_after, color='red', line_width=10)
        p2.show()



=== DIAGNOSTIC ===
Original mesh is_watertight: False
Filled mesh (trimesh) is_watertight: True
Filled mesh volume: 13235775572.906181

Holes before filling: 2223 hole segments
Holes after filling: 4446 hole segments

✓ SUCCESS: Mesh is watertight according to trimesh!
  (The remaining holes detected by MeshFix are likely topological artifacts)
  Your mesh is ready for use.
  Saved to: equimesh_n100lobe_w75_h25_watertight.stl
